In [172]:
import numpy as np
import xarray as xr
import scipy.io as sio
# from plot_icebergshape import plot_icebergshape
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, interp2d
from matplotlib import cm,colors
import pickle
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import pandas as pd


In [173]:
melange_area_path = '/home/m484s199/iceberg_py/dev/geoms/helheim/melange_area/'
mel_list = sorted( [file for file in os.listdir(melange_area_path) if file.endswith('gpkg') ] )
date_list = [gpd.pd.to_datetime(file[8:18]) for file in mel_list]
gdf_list = [gpd.read_file(f'{melange_area_path}{file}') for file in mel_list]
    


In [174]:
mel_list

['helhiem_2016-04-24_area.gpkg',
 'helhiem_2018-05-04_area.gpkg',
 'helhiem_2020-09-03_area.gpkg',
 'helhiem_2023-07-27_area.gpkg',
 'helhiem_2024-08-22_area.gpkg']

In [175]:
date_list

[Timestamp('2016-04-24 00:00:00'),
 Timestamp('2018-05-04 00:00:00'),
 Timestamp('2020-09-03 00:00:00'),
 Timestamp('2023-07-27 00:00:00'),
 Timestamp('2024-08-22 00:00:00')]

In [176]:
gdf_list[0].area[0]

np.float64(132528531.77252682)

In [177]:
area_dict = {date:gdf.area[0] for date, gdf in zip(date_list, gdf_list)}
area_dict

{Timestamp('2016-04-24 00:00:00'): np.float64(132528531.77252682),
 Timestamp('2018-05-04 00:00:00'): np.float64(142934146.47607896),
 Timestamp('2020-09-03 00:00:00'): np.float64(141325864.68025035),
 Timestamp('2023-07-27 00:00:00'): np.float64(142934146.47607896),
 Timestamp('2024-08-22 00:00:00'): np.float64(134061538.63418496)}

In [178]:
temp_dict = {'min':4.0,
            'avg': 5.4,
            'max': 6.3} #from CTD AW average TF

urel_dict = {'min':0.07,
            'avg': 0.13,
            'max': 0.20} #model vel runs

In [179]:
run_type = 'avg'

dt = 50
psw = 1024 #kg m3
csw = 3974 #J kg-1 C-1
day2sec = 86400
depth = 450
temp = temp_dict[run_type]
coeff_1_path = f'/home/m484s199/iceberg_py/data/iceberg_model_output_melt_fix/helheim/{run_type}/'
coeff_1_list = sorted([nc for nc in os.listdir(coeff_1_path) if nc.endswith('nc')])

def Qaw_calc(area, dt = dt, psw = psw, csw = csw, depth = depth, temp = temp):
    
    vol = area * depth
    
    Qaw = psw * csw * ( (vol * temp) / (dt * day2sec) )

    return Qaw

dQ_dt_HEL_CTD_avg_dict = {date:Qaw_calc(vol) for date, vol in area_dict.items()}

# dQ_dt_HEL_CTD_avg = psw * csw * ( (Volume_test * 5.4) / (dt * day2sec) )

os.chdir(coeff_1_path)

cols = ['coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'melt_rate_mday', 'percentage']

In [180]:
dQ_dt_HEL_CTD_avg_dict

{Timestamp('2016-04-24 00:00:00'): np.float64(303360989912.0764),
 Timestamp('2018-05-04 00:00:00'): np.float64(327179691703.2602),
 Timestamp('2020-09-03 00:00:00'): np.float64(323498296073.8454),
 Timestamp('2023-07-27 00:00:00'): np.float64(327179691703.2602),
 Timestamp('2024-08-22 00:00:00'): np.float64(306870079410.5766)}

In [181]:
# dQ_dt_HEL_CTD_avg/1e9

In [182]:
cols = ['date', 'coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'percentage']
coef_1_dict = {}

series_list1 = []
for i,nc in enumerate(coeff_1_list):
    
    Qib = xr.open_dataset(f'{coeff_1_path}{nc}')
    Qib_val = Qib.Qib.data
    date = nc[:10]
    date = pd.to_datetime(date)
    TF = nc.split('_')[6]
    urel_val = nc.split('_')[-1].split('.')[1]
    
    # print(f'coeff 1: {percentage:.2f}')
    print(f'{date}')
    coef_1_dict['date'] = date
    coef_1_dict['coeff'] = 1
    coef_1_dict['urel'] = urel_dict[run_type]
    coef_1_dict['tf'] = temp_dict[run_type]
    coef_1_dict['dt'] = dt
    coef_1_dict['Qib'] = (Qib_val/1e11)
    # coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg #(dQ_dt_dict[TF])
    coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg_dict[date]
    
    coef_1_dict['melt_rate_avg_m3s'] = Qib.melt_rate_integrated.data
    
    coef_1_dict['percentage'] = f'{(Qib_val/dQ_dt_HEL_CTD_avg_dict[date])*100:.2f}' #f'{(Qib_val/dQ_dt_dict[TF])*100:.2f}'
    
    series = pd.Series(coef_1_dict)
    series_list1.append(series)

df_50 = pd.DataFrame(series_list1, columns=cols)   



2016-04-24 00:00:00
2018-05-04 00:00:00
2020-09-03 00:00:00
2023-07-27 00:00:00
2024-08-22 00:00:00


In [183]:
!pwd

/home/m484s199/iceberg_py/data/iceberg_model_output_melt_fix/helheim/avg


In [184]:
nc.split('_')

['2024-08-22', 'helheim', 'coeff', '1', 'CTD', 'constant', 'UREL', '13.nc']

In [185]:
df_50

,date,coeff,urel,tf,dt,Qib,Qaww,melt_rate_avg_m3s,percentage
0,2016-04-24,1,0.13,5.4,50,0.135215,3.033610e+11,40.362581332675035,4.46
1,2018-05-04,1,0.13,5.4,50,0.142444,3.271797e+11,42.5205925889621,4.35
2,2020-09-03,1,0.13,5.4,50,0.165533,3.234983e+11,49.412896865551375,5.12
3,2023-07-27,1,0.13,5.4,50,0.306006,3.271797e+11,91.34511465073093,9.35
4,2024-08-22,1,0.13,5.4,50,0.236273,3.068701e+11,70.52933101542344,7.70


In [186]:
df_50['percentage'].astype(np.float64).mean()

np.float64(6.196)

In [168]:
run_type = 'avg'

dt = 150
psw = 1024 #kg m3
csw = 3974 #J kg-1 C-1
day2sec = 86400
depth = 450
temp = temp_dict[run_type]


def Qaw_calc(area, dt = dt, psw = psw, csw = csw, depth = depth, temp = temp):
    
    vol = area * depth
    
    Qaw = psw * csw * ( (vol * temp) / (dt * day2sec) )

    return Qaw

dQ_dt_HEL_CTD_avg_dict = {date:Qaw_calc(vol) for date, vol in area_dict.items()}

# dQ_dt_HEL_CTD_avg = psw * csw * ( (Volume_test * 5.4) / (dt * day2sec) )

os.chdir(coeff_1_path)

cols = ['coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'melt_rate_mday', 'percentage']

In [169]:
cols = ['date', 'coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'percentage']
coef_1_dict = {}

series_list1 = []
for i,nc in enumerate(coeff_1_list):
    
    Qib = xr.open_dataset(f'{coeff_1_path}{nc}')
    Qib_val = Qib.Qib.data
    date = nc[:10]
    date = pd.to_datetime(date)
    TF = nc.split('_')[6]
    urel_val = nc.split('_')[-1].split('.')[1]
    
    # print(f'coeff 1: {percentage:.2f}')
    print(f'{date}')
    coef_1_dict['date'] = date
    coef_1_dict['coeff'] = 1
    coef_1_dict['urel'] = urel_dict[run_type]
    coef_1_dict['tf'] = temp_dict[run_type]
    coef_1_dict['dt'] = dt
    coef_1_dict['Qib'] = (Qib_val/1e11)
    # coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg #(dQ_dt_dict[TF])
    coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg_dict[date]
    
    coef_1_dict['melt_rate_avg_m3s'] = Qib.melt_rate_integrated.data
    
    coef_1_dict['percentage'] = f'{(Qib_val/dQ_dt_HEL_CTD_avg_dict[date])*100:.2f}' #f'{(Qib_val/dQ_dt_dict[TF])*100:.2f}'
    
    series = pd.Series(coef_1_dict)
    series_list1.append(series)

df_150 = pd.DataFrame(series_list1, columns=cols)   



2016-04-24 00:00:00
2018-05-04 00:00:00
2020-09-03 00:00:00
2023-07-27 00:00:00
2024-08-22 00:00:00


In [170]:
df_150

,date,coeff,urel,tf,dt,Qib,Qaww,melt_rate_avg_m3s,percentage
0,2016-04-24,1,0.13,5.4,150,0.135215,1.011203e+11,40.362581332675035,13.37
1,2018-05-04,1,0.13,5.4,150,0.142444,1.090599e+11,42.5205925889621,13.06
2,2020-09-03,1,0.13,5.4,150,0.165533,1.078328e+11,49.412896865551375,15.35
3,2023-07-27,1,0.13,5.4,150,0.306006,1.090599e+11,91.34511465073093,28.06
4,2024-08-22,1,0.13,5.4,150,0.236273,1.022900e+11,70.52933101542344,23.10


In [187]:
df_150['percentage'].astype(np.float64).mean()

np.float64(18.588)

In [193]:
os.chdir('/home/m484s199/iceberg_py/')
op = './data/csv/'
op50 = f'{op}AVG_dt50.csv'
df_50.to_csv(op50,index=False)

op150 = f'{op}AVG_dt150.csv'
df_150.to_csv(op150, index=False)